# Session8: Fine Tunning:
This session will be about Fine-Tunning concepts and how to apply it practically using a Pre-trained model BERT ( or as in our case "distilbert" (which is just a lighter version of it)).

## First Example:

Here's some explainations about the libraries we have used:

`from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments`:

- `AutoTokenizer`: Automatically loads the appropriate tokenizer for a given pretrained model.

- `AutoModelForSequenceClassification`: Automatically loads a pretrained model adapted for sequence classification tasks (like sentiment analysis). loads a pretrained model (like BERT) and automatically adds a classification head on top, typically a small neural network that takes the model's output and maps it to our specific number of labels (e.g., 2 for positive/negative, 3 for positive/neutral/negative).

- `Trainer`: A high-level API for training and evaluating Hugging Face models.  It's designed to make training, evaluating, and saving transformer models extremely simple.

- `TrainingArguments`: A class to configure training parameters (learning rate, batch size, epochs, etc.).

`from torch.utils.data import Dataset`:  Imports the `Dataset` abstract base class from PyTorch's data utilities. This class is used to create custom datasets by overriding `__len__` and `__getitem__` methods to feed data into PyTorch models. PyTorch's `Dataset` class is an abstract base class - it provides a template, but we must fill in the details for our specific data format (CSV, JSON, images in folders, database records, etc.). This import allows us to create our own dataset class that can be passed to the `Trainer`. The `Trainer` expects a dataset object that has `__len__` and `__getitem__ `methods . 

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" #Sets an environment variable KMP_DUPLICATE_LIB_OK to "TRUE", in order to prevent library duplication errors , which particularly happen when using PyTorch with certain configurations.

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

In [2]:
# Install required libraries
!pip install accelerate>=1.1.0
!pip install transformers
!pip install torch

Key Points:

Custom dataset = we define how to load OUR data format.

`__init__` = Load and prepare your data (CSV, JSON, database, etc.).

`__len__` = Return total number of examples.

`__getitem__` = Return one example (index idx) as tensors.

The `Trainer` will automatically loop through our dataset using these methods.

In [3]:
#1. Creating a customized Dataset:

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        # Load and prepare our data
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        # returning a dict that contains input_ids, attention_mask and labels
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    
    def __len__(self):
        # Return total number of samples
        return len(self.labels)

### Loading the pre-trained model:
In the two cells below, we're displaying 2 ways to load our model: 
-  In the first cell we're loading our model "distilbert-base-uncased" online, this will load the model from Hugging Face Hub (this will take some time to load , based on your internet connection speed.).
  
-  In the second cell we're loading the model from a local directory `"./model/distilbert_model"` (offline), we have already downloaded the same model and it's other configration files in the `model` folder , and we're loading it locally from it. We recommend you to use this approach to avoid bad connections problems , and to avoid loading the model every time you run this session.

You can download this model from this link: !(https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/tree/main) then download the following files: `config.json`,`model.safetensors`, `tokenizer_config.json`, `vocab.txt`. Put them all in one folder , and then you can load it to your jupyter notebook.  

________________

`AutoTokenizer.from_pretrained()`: Automatically detects and loads the correct tokenizer architecture for the model:

- `"distilbert-base-uncased"`: The model identifier (a smaller, faster version of BERT).

    - distilbert: Distilled version of BERT (40% smaller, 60% faster, 97% performance).

    - base: Standard size (not large or small).

    - uncased: Converts all text to lowercase and ignores case sensitivity.
 
What the tokenizer does:

- Converts text into tokens (subwords or words).

- Adds special tokens like [CLS] (classification token) and [SEP] (separator).

- Converts tokens to numerical IDs.

- Creates attention masks (which tokens are real vs padding).

- Handles padding and truncation.


`AutoModelForSequenceClassification.from_pretrained()`: Loads a pretrained model and automatically adds a sequence classification head:

- `"distilbert-base-uncased"`: Same base model as above

- `num_labels=2`: Specifies this is a binary classification task (2 output classes)

In [ ]:
# The first approach: Loadingt the model online
# Note You only run this cell or the next one, NOT both of them!

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # load a pretrained DistilBERT model. 
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) # loads its corresponding tokenizer.

In [5]:
# The second approach : loading the model locally (offline)
# Note You only run this cell or the previous one, NOT both of them!
# 2. Loading the tokenizer and the model:

tokenizer = AutoTokenizer.from_pretrained("./model/distilbert_model") # load a pretrained DistilBERT model. 
model = AutoModelForSequenceClassification.from_pretrained("./model/distilbert_model", num_labels=2) # loads its corresponding tokenizer

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [7]:
# 4. Preparing the dataset:

texts = [
"This movie is great!",
"I hate this film",
"Wonderful acting",
"Terrible plot"
]

# Creating the labels that coresspond to the texts above:

labels = [1, 0, 1, 0] # 1: mens positive, 2: means negative

`truncation=True`: Cuts off texts that are too long for the model's maximum length, DistilBERT's max length is typically 512 tokens, If a text has >512 tokens, it gets truncated to 512, without this, we'd get an error for long texts.

`padding=True`: Makes all sequences the same length by adding special [PAD] tokens to shorter texts, batching requires all tensors to have the same dimensions, so shorter texts get padded to match the longest text in the batch.

`return_tensors="pt"`: Returns PyTorch tensors instead of Python lists:

- `"pt"` = PyTorch, `"tf"` = TensorFlow, `"np"` = NumPy.

- This makes the output directly usable with your PyTorch model!

In [8]:
# 4. Encoding the data:
#  tokenizes all our texts at once and converts them into PyTorch tensors
encodings = tokenizer(texts, trunca5tion=True, padding=True, return_tensors="pt")

In [9]:
# 5. Creating a customized dataset:
# # This encodings dictionary is what our TextDataset expects
train_dataset = TextDataset(encodings, labels)

`output_dir="./my_model"`: Specifies where to save the trained model and training checkpoints, **Note**: The directory will be created automatically if it doesn't exist. 

`num_train_epochs=3`: Number of complete passes through the entire training dataset. **NOTE**: Typical values: 2-5 for fine-tuning transformer models (more can cause overfitting).

`per_device_train_batch_size=2`: Number of samples processed together on each GPU/CPU before updating model.

`logging_steps=10`: Logs training metrics (loss, learning rate, etc.) every 10 steps.

In [10]:
# 6. Setting Training Parameters:

training_args = TrainingArguments(
    output_dir="./my_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=10,
)

In [11]:
# 7. Start the Training Process:
# creating the Trainer and start the  training process. 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset #  Now it passes a valid Dataset object
)

trainer.train() # start training

C:\Users\asus--2018\anaconda3\envs\nlp_course_2026\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=0.00024039057704309622, metrics={'train_runtime': 15.7044, 'train_samples_per_second': 0.764, 'train_steps_per_second': 0.382, 'total_flos': 21732932592.0, 'train_loss': 0.00024039057704309622, 'epoch': 3.0})

In [12]:
# 8. Saving the model:
# save your fine-tuned model and tokenizer to disk so you can reuse them later without retraining.
model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_model\\tokenizer_config.json', './my_model\\tokenizer.json')

# Second Example:
In this example we're using the the same model , on the same task (sentiment analysis) except for some differences in some details such as the way that we have defined our dataset and it's labels. 

Try to figuare out these differences by yourself, and how we handeled them!

In [15]:
!pip install datasets

   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---

In [16]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [17]:
tokenizer = AutoTokenizer.from_pretrained("./model/distilbert_model")
model = AutoModelForSequenceClassification.from_pretrained("./model/distilbert_model", num_labels=2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [18]:

data = {
"text": [
"This movie is great!",
"I hate this film",
"Wonderful acting",
"Terrible plot"
],
"label": [1, 0, 1, 0]
}

In [19]:
dataset = Dataset.from_dict(data)

In [21]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True)

In [22]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [23]:
training_args = TrainingArguments(
    output_dir="./my_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=10,
)
    

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset, # يعمل مباشرة
    processing_class=tokenizer,
)

trainer.train()

C:\Users\asus--2018\anaconda3\envs\nlp_course_2026\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=0.00024039057704309622, metrics={'train_runtime': 5.6531, 'train_samples_per_second': 2.123, 'train_steps_per_second': 1.061, 'total_flos': 21732932592.0, 'train_loss': 0.00024039057704309622, 'epoch': 3.0})

In [25]:
model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_model\\tokenizer_config.json', './my_model\\tokenizer.json')